In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\DR.KSS_2025.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,328.0,355.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,318.0,326.0,125.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,371.0,286.0,156.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,378.0,264.0,148.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,339.0,284.0,119.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,335.0,210.0,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,296.0,156.0,202.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,297.0,152.0,158.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,357.0,227.0,209.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,397.0,271.0,197.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,328.0,191.5,126.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,318.0,326.0,125.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,371.0,286.0,156.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,378.0,264.0,148.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,339.0,284.0,119.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,335.0,210.0,124.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,296.0,156.0,202.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,297.0,152.0,158.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,357.0,227.0,209.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,397.0,271.0,197.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_ml_ready = df_ml_ready.dropna(axis=1, how='all')


In [9]:
df_ml_ready

,Day,January,February,March,April
0,1,328.0,191.5,126.0,100.0
1,2,318.0,326.0,125.0,100.0
2,3,371.0,286.0,156.0,100.0
3,4,378.0,264.0,148.0,100.0
4,5,339.0,284.0,119.0,100.0
5,6,335.0,210.0,124.0,100.0
6,7,296.0,156.0,202.0,100.0
7,8,297.0,152.0,158.0,100.0
8,9,357.0,227.0,209.0,100.0
9,10,397.0,271.0,197.0,100.0
